## Simple implemenation of Bifrost


**Installing Env Variables**

In [2]:
import os
from dotenv import load_dotenv

# Load secrets from .env — never hardcode API keys in this notebook.
# Re-run this cell after editing .env (no kernel restart needed).
load_dotenv(override=True)

REQUIRED_ENV_VARS = (
    "OPENAI_API_KEY",
    "GROQ_API_KEY",
    "BIFROST_OPENAI_API_KEY",
    "BIFROST_GROQ_API_KEY",
    "BIFROST_BASE_URL"
)

missing = [name for name in REQUIRED_ENV_VARS if not os.getenv(name)]
if missing:
    raise ValueError(
        "Missing required environment variables: "
        + ", ".join(missing)
        + ". Copy .env.example to .env and set your values."
    )

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
BIFROST_OPENAI_API_KEY = os.getenv("BIFROST_OPENAI_API_KEY")
BIFROST_GROQ_API_KEY = os.getenv("BIFROST_GROQ_API_KEY")
BIFROST_GROQ_API_KEY = os.getenv("BIFROST_GROQ_API_KEY")
BIFROST_BASE_URL = os.getenv("BIFROST_BASE_URL")

print("Environment loaded.")


Environment loaded.


**Health Check**

In [3]:
import httpx
try:
    health = httpx.get(f"{BIFROST_BASE_URL}/health")
    print(health.json())
except Exception as e:
    print(f"Error: {e}")


{'components': {'db_pings': 'ok'}, 'status': 'ok'}


### Reusable Text Prompts and Utilits

In [4]:
### Reusable Text Prompts and Utilits
TEST_PROMPTS = {
    "simple": "What is the capital of France?",
    "reasoning": "Explain the difference between RAG and fine-tuning in 3 bullet points.",
    "code": "Write a Python function that validates an email address using regex.",
    "duplicate1": "What is LangChain used for?",
    "duplicate2": "What is LangChain primarily used for?",
    "deepwiki": "What are the stream modes in the new langgraph version? Use the deepwiki",
    "tavily": "Search the web for the latest news about Groq AI and summarize the top"
}

import time
def time_caputer(func):
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        return result, end_time - start_time
    return wrapper  # <-- this was missing


CHATOPENAI_MODEL = "openai/gpt-4o-mini"
CHATOPENAI_FALLBACK_MODEL = "openai/gpt-5.6-luna"

# provider/model format — groq/ routes to Groq, openai/ routes to OpenAI
GROQ_MODEL = "groq/openai/gpt-oss-20b"
GROQ_FALLBACK_MODEL = "openai/gpt-4o-mini"

MODEL_DEFAULT_FALLBACKS = {
    CHATOPENAI_MODEL: CHATOPENAI_FALLBACK_MODEL,
    GROQ_MODEL: GROQ_FALLBACK_MODEL,
}


**Bad Approach Calling direct model**

No caching , routing , virtual keys can be applied , for this we need to make extra efforts 

In [5]:
from langchain_openai import ChatOpenAI 

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    streaming=True
)

@time_caputer
def getResponse(prompt):
    response_content = ""
    for chunk in llm.stream(prompt):
        text = chunk.content or ""
        response_content += text
        print(text, end="", flush=True)
    return response_content

response, elapsed = getResponse("Give 200 words essay on OpenTelemetry")
print(f"\nElapsed: {elapsed:.2f}s")

OpenTelemetry is an open-source observability framework designed to provide a standardized way to collect, process, and export telemetry data from applications. It encompasses metrics, logs, and traces, enabling developers and operators to gain insights into the performance and behavior of their systems. As cloud-native architectures and microservices become increasingly prevalent, the need for effective observability tools has grown, making OpenTelemetry a vital component in modern software development.

One of the key advantages of OpenTelemetry is its vendor-agnostic nature, allowing organizations to instrument their applications without being locked into a specific monitoring solution. This flexibility enables teams to choose the best tools for their needs while maintaining a consistent approach to data collection. OpenTelemetry supports multiple programming languages, including Java, Python, Go, and JavaScript, making it accessible to a wide range of developers.

By providing a un

## Using the BiFrost LLM Gateway

In [ ]:
from langchain_openai import ChatOpenAI 

llm = ChatOpenAI(
    model=CHATOPENAI_MODEL,              # "openai/gpt-4o-mini"
    base_url=f"{BIFROST_BASE_URL}/langchain",
    api_key=BIFROST_GROQ_API_KEY,      # not GROQ key
    temperature=0,
    streaming=True,
)

@time_caputer
def getResponse(prompt):
    response_content = ""
    for chunk in llm.stream(prompt):
        text = chunk.content or ""
        response_content += text
        print(text, end="", flush=True)
    return response_content

response, elapsed = getResponse("Give 200 words essay on OpenTelemetry")
print(f"\nElapsed: {elapsed:.2f}s")

OpenTelemetry is an open-source observability framework designed to provide a standardized way to collect, process, and export telemetry data from applications. It encompasses three primary types of telemetry: traces, metrics, and logs, enabling developers to gain comprehensive insights into their systems' performance and behavior. By offering a unified approach, OpenTelemetry simplifies the integration of observability into diverse programming languages and platforms, fostering consistency across different environments.

One of the key advantages of OpenTelemetry is its vendor-agnostic nature, allowing organizations to choose their preferred back-end systems for data analysis without being locked into a specific vendor's ecosystem. This flexibility encourages innovation and enables teams to adapt their observability strategies as their needs evolve.

Moreover, OpenTelemetry promotes collaboration among developers, operators, and stakeholders by providing a common language for discussi

**Bifrost Helper function**


In [ ]:
import json as _json

from openai import OpenAI

DEFAULT_CACHE_KEY = "notebook-demo"

_bifrost_clients: dict[str, OpenAI] = {}


def _provider_from_model(model: str) -> str:
    return model.split("/", 1)[0] if "/" in model else "openai"


def _bifrost_client(model: str) -> OpenAI:
    provider = _provider_from_model(model)
    api_key = BIFROST_GROQ_API_KEY if provider == "groq" else BIFROST_GROQ_API_KEY

    if provider not in _bifrost_clients:
        _bifrost_clients[provider] = OpenAI(
            base_url=f"{BIFROST_BASE_URL}/openai",
            api_key=api_key,
        )
    return _bifrost_clients[provider]


def _resolve_fallbacks(model: str, fallback_model: str | None, fallback_models: list[str] | None) -> list[str]:
    if fallback_models:
        return list(fallback_models)
    if fallback_model:
        return [fallback_model]
    return [MODEL_DEFAULT_FALLBACKS[model]] if model in MODEL_DEFAULT_FALLBACKS else []


def call_bifrost(
    prompt: str,
    model: str,
    fallback_model: str | None = None,
    fallback_models: list[str] | None = None,
    cache_key: str | None = DEFAULT_CACHE_KEY,
    cache_type: str | None = "direct",
):
    client = _bifrost_client(model)
    fallbacks = _resolve_fallbacks(model, fallback_model, fallback_models)

    headers = {}
    if cache_key:
        headers["x-bf-cache-key"] = cache_key
    if cache_type:
        headers["x-bf-cache-type"] = cache_type

    request_kwargs = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
    }
    if headers:
        request_kwargs["extra_headers"] = headers
    if fallbacks:
        # Bifrost reads fallbacks from the JSON body (not only from headers).
        request_kwargs["extra_body"] = {"fallbacks": fallbacks}

    raw_response = client.chat.completions.with_raw_response.create(**request_kwargs)

    response = raw_response.parse()
    body = _json.loads(raw_response.text)
    extra = body.get("extra_fields", {})
    cache_debug = extra.get("cache_debug", {})

    fallback_index = raw_response.headers.get("x-bifrost-fallback-index")
    print(
        "provider="
        f"{extra.get('provider')} "
        f"resolved_model={extra.get('resolved_model_used')} "
        f"fallback_index={fallback_index or '0 (primary)'} "
        f"cache_hit={cache_debug.get('cache_hit')} "
        f"hit_type={cache_debug.get('hit_type')}"
    )
    if fallbacks:
        print(f"fallbacks={fallbacks}")

    return response.choices[0].message.content, body


In [8]:
content1, meta1 = call_bifrost(
    "why people fall in Love and after some time hate the same person",
    GROQ_MODEL,
    fallback_model=GROQ_FALLBACK_MODEL,
)
print(content1)


provider=groq resolved_model=openai/gpt-oss-20b fallback_index=0 (primary) cache_hit=False hit_type=None
fallbacks=['openai/gpt-4o-mini']
### Why the “Love‑then‑Hate” Cycle Happens

It feels like a dramatic flip‑flop, but it’s actually a common, very human pattern.  
Below is a mix of psychological science, everyday experience, and a few practical tips that help make sense of the transition.

---

## 1. The “First‑Love” Phase – A Cocktail of Idealization & Hormones

| What Happens | Why It Feels Like Love |
|--------------|-----------------------|
| **Selective Perception** | When you’re attracted, you notice the good, filter the bad. Your brain is wired to *seek* a partner who “fits” you. |
| **Hormonal Boost** | Dopamine, oxytocin, and adrenaline spike. These chemicals give you that euphoric “high” and a sense that everything is “right.” |
| **Romantic Narratives** | Media, stories, and past experiences shape your expectations of what a “perfect” partner should look like. |

**Result

### **MCP with the Bifrost**

**Naming the Tools**

In [ ]:
class BifrostApiError(Exception):
    """Readable Bifrost / HTTP error with status + hint."""

    def __init__(self, message: str, status_code: int | None = None, hint: str | None = None):
        self.status_code = status_code
        self.hint = hint
        parts = [message]
        if hint:
            parts.append(f"Hint: {hint}")
        super().__init__("\n".join(parts))


def _status_hint(status_code: int, body: dict) -> str | None:
    err = body.get("error") or {}
    code = (err.get("code") or err.get("type") or "").lower()
    msg = (err.get("message") or "").lower()

    if status_code == 403:
        if "provider" in msg or code == "provider_blocked":
            return (
                "Virtual key and model provider must match "
                "(openai key → openai/..., groq key → groq/...)."
            )
        if "mcp" in msg:
            return "Add mcp_configs for this client on the virtual key in Bifrost UI."
        return "Check virtual key provider_configs / mcp_configs in Bifrost."
    if status_code == 401:
        return "Check BIFROST_*_API_KEY in .env matches the virtual key in Bifrost."
    if status_code == 429:
        return "Rate limited. Wait a bit, use one MCP client, or relax limits in Bifrost."
    if status_code >= 500:
        return "Bifrost or upstream provider error. Check Bifrost logs / LLM Logs in UI."
    return None


def _raise_for_bifrost_error(response: httpx.Response) -> None:
    if response.is_success:
        return

    body: dict = {}
    try:
        body = response.json()
    except ValueError:
        pass

    err = body.get("error") or {}
    detail = (
        err.get("message")
        or body.get("message")
        or response.text
        or response.reason_phrase
    )
    hint = _status_hint(response.status_code, body)
    raise BifrostApiError(
        f"Bifrost request failed ({response.status_code}): {detail}",
        status_code=response.status_code,
        hint=hint,
    )


def _api_key_for_model(model: str) -> str:
    provider = model.split("/", 1)[0]
    if provider == "groq":
        return BIFROST_GROQ_API_KEY
    if provider == "openai":
        return BIFROST_OPENAI_API_KEY
    raise ValueError(f"No Bifrost virtual key configured for provider '{provider}'")


In [16]:
def fetch_mcp_clients() -> list[str]:
    try:
        response = httpx.get(f"{BIFROST_BASE_URL}/api/mcp/clients", timeout=5)
        _raise_for_bifrost_error(response)
    except httpx.RequestError as e:
        raise BifrostApiError(f"Could not reach Bifrost at {BIFROST_BASE_URL}: {e}") from e

    clients = response.json().get("clients", [])
    return [
        client["config"]["name"]
        for client in clients
        if client.get("config", {}).get("name")
    ]


try:
    registered = fetch_mcp_clients()
    print(registered)
except BifrostApiError as e:
    print(f"Error: {e}")
    if e.status_code:
        print(f"HTTP status: {e.status_code}")

['BifrostWikiDeep', 'Travily']


In [28]:
def call_with_tools(
    prompt: str,
    mcp_clients: list[str] | set[str] | None = None,
    model: str = "groq/openai/gpt-oss-20b",
) -> dict:
    clients = list(mcp_clients) if mcp_clients is not None else fetch_mcp_clients()
    print(f"MCP clients: {clients}")
    if not clients:
        raise BifrostApiError("No MCP clients registered in Bifrost.")

    headers = {
        "Authorization": f"Bearer {_api_key_for_model(model)}",
        "Content-Type": "application/json",
        "x-bf-mcp-include-clients": ",".join(clients),
    }

    try:
        response = httpx.post(
            f"{BIFROST_BASE_URL}/v1/chat/completions",
            headers=headers,
            json={
                "model": model,
                "messages": [{"role": "user", "content": prompt}],
            },
            timeout=60,
        )
        _raise_for_bifrost_error(response)
    except httpx.TimeoutException:
        raise BifrostApiError(
            "Request timed out after 60s. MCP + LLM can be slow; retry or use one client."
        )
    except httpx.RequestError as e:
        raise BifrostApiError(f"Network error talking to Bifrost: {e}") from e

    return response.json()


try:
    result = call_with_tools(
        "Tell me Indian Indepence from wiki peadia use BifrostWikiDeep?",
        mcp_clients=["BifrostWikiDeep"],
    )
    message = result["choices"][0]["message"]
    print(message.get("content") or message.get("tool_calls"))
except BifrostApiError as e:
    print(f"Error: {e}")
    if e.status_code:
        print(f"HTTP status: {e.status_code}")
except ValueError as e:
    print(f"Config error: {e}")

['BifrostWikiDeep', 'Travily']
India gained independence from British rule on 15 August 1947, ending almost two centuries of colonial domination. The transition followed the Indian Independence Act of 1947, which partitioned the sub‑continent into the sovereign states of India and Pakistan. The movement was led by figures such as Mahatma Gandhi, Jawaharlal Nehru, and Sardar Vallabhbhai Patel, and it involved mass civil disobedience, non‑violent protest, and political negotiation with the British government.  

*Source: [Wikipedia – Indian Independence](https://en.wikipedia.org/wiki/Indian_Independence)*
